# 02.4 Residence Time Distributions: Exercises

## Contents

**Tools:** CADET-GUI widgets: unit parameters and simulation; CADET-Process (Python).

**Python used:** Simple calculations, NumPy arrays and numerical integration with supplied code; object creation and connections.

**Practises:** `Cstr`, `TubularReactor`, system delay, system volume and porosity correction.

1. Delay and spreading
2. System volume
3. Column porosity correction (optional)
4. The instrument in Python (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Characterize the instrument with the column bypassed, then use its measured delay to interpret column retention. Exercises marked **optional** extend the main tasks and can be skipped.

## Starter system

The model is **Inlet → Mixer → Tubing → Outlet**, with the column bypassed. Flow is **6 mL/min**; mixer and tubing volumes are **1 and 2 mL**. Tracer enters as a **5 s pulse**. A synthetic tracer-pulse dataset provides the reference detector curve for this experiment.

```{figure} ./resources/system_bypass.svg
:width: 80%
:align: center
:alt: Inlet connected to a 1 mL mixer, 2 mL tubing and Outlet, with the column bypassed.
```

```{tip}
Run the cell below once with **Shift+Enter** to load the supplied system model for this exercise.
```

In [ ]:
# Load the supplied column-bypass model and its simulation widget.
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from cadetgui import SimulationWidget
%run ../resources/first_steps/lesson_setup.py
system_component_system, system_units, system_sheet, system_process = rtd_system_model()

The file `resources/measured_system_pulse.csv` contains detector concentration versus time for the column-bypass experiment. The next cell loads these synthetic data as the reference curve **system pulse**; time is in seconds.

In [ ]:
from CADETProcess.reference import ReferenceIO
pulse_data = np.loadtxt("resources/measured_system_pulse.csv", delimiter=",", skiprows=1)
# CSV columns: 0 = time / s, 1 = tracer concentration / mol/m³
system_reference = ReferenceIO("system pulse", pulse_data[:, 0], pulse_data[:, 1])

## Exercise 1: distinguish delay and spreading

| Setting | Value | SI value |
|---|---:|---:|
| Flow rate $Q$ | 6 mL/min | `1e-7` m³/s |
| Mixer volume $V_\text{mixer}$ | 1 mL | `1e-6` m³ |
| Tubing volume $V_\text{tubing}$ | 2 mL | `2e-6` m³ |
| Tracer pulse | 5 s | |

Use $\tau=V/Q$ to calculate the mean delay added by mixer and tubing in the answer cell. Give both delays in **seconds**.

In [ ]:
# Mixer and tubing delays in seconds


Now compare the signals in the simulation panel below:

1. Click **Run simulation**.
2. Under **Show**, select `mixer`, then `tubing`: each shows the signal leaving that unit. Which unit adds a tail, and which mainly shifts the pulse?
3. Select `outlet`: this is the simulated detector signal. Compare it with the dashed **system pulse**, the reference curve loaded from the CSV file.

```{tip}
Run the next cell with **Shift+Enter** to open the unit editor and the simulation panel.
```

In [ ]:
compact_unit_widget(system_component_system, system_units, create=False).display()
SimulationWidget(system_process, reference=system_reference).display()

```{dropdown} Answer
Mixer and tubing add mean delays of 10 and 20 s. The mixer gives a tail; tubing mainly delays the pulse. The complete response includes both contributions and the finite injection width.
```

## Exercise 2: system volume

The supplied code calculates the centroid of the reference detector curve, $\bar t=\int tc(t)dt/\int c(t)dt$. Subtract the inlet centroid, 2.5 s, then calculate $V_\mathrm{sys}=Q\tau_\mathrm{sys}$ in the answer cell.

Record this system delay for the column experiment in Exercise 3. Does the total volume alone tell you how much belongs to mixer or tubing?

In [ ]:
# Supplied data calculation: mean time at the outlet.
t, signal = pulse_data.T
centroid = np.trapezoid(t * signal, t) / np.trapezoid(signal, t)
print(f"Outlet centroid: {centroid:.2f} s")

In [ ]:
# System delay and volume


```{dropdown} Interpretation
The total system volume outside the column is about 3 mL. The centroid alone cannot divide it between mixer and tubing; the signal shape provides additional information. Carry the system settings forward when fitting column transport.
```

## Exercise 3 (optional): correct the column porosity

The column is now reconnected: the same mixer and tubing feed a **10 × 2 cm column** at **6 mL/min**. A nonbinding tracer is supplied continuously at **1 mol/m³**, starting at time zero. The synthetic breakthrough includes instrument and column transport. The supplied code calculates its mean delay, $\tau_\mathrm{measured}=\int(1-c_\mathrm{out}/c_\mathrm{feed})dt$.

Use $\varepsilon_t=\tau Q/V$ to estimate **apparent porosity** from the full measured delay. Then subtract the system delay (checkpoint: 30 s) and estimate the tracer-accessible column porosity again. Why is the corrected value smaller?

In [ ]:
# Supplied data calculation: mean breakthrough delay and column volume.
step_data = np.loadtxt("../resources/first_steps/measured_breakthrough.csv", delimiter=",", skiprows=1)
t, concentration = step_data.T
tau_measured = np.trapezoid(1 - concentration, t)
column_volume = np.pi * 0.01**2 * 0.1
print(f"Mean breakthrough delay: {tau_measured:.2f} s")

In [ ]:
# Apparent and corrected porosity


```{dropdown} Answer
Correcting the 30 s system delay changes the porosity from about 0.445 to 0.35. The measured delay includes residence time in the mixer and tubing. Keep the characterized system in the model when fitting the column, then keep transport fixed when examining protein binding.
```

```{note}
Subtracting the system delay corrects the mean retention time, but not the shape of the curve: the mixer and tubing also spread the pulse, so the measured curve is wider than the column alone would make it. To fit the column's axial dispersion, simulate mixer, tubing and column together and compare that complete model with the measured curve.
```

## Exercise 4 (optional): the instrument in Python

Build the column-bypass instrument in Python, as in the lesson, and compare its detector signal with the measured pulse. The supplied cell creates the tracer, inlet and outlet in a new flow sheet.

In [ ]:
# Tracer, inlet and outlet in a new flow sheet; mixer and tubing are added below.
from CADETProcess.processModel import (
    ComponentSystem, Inlet, Outlet, FlowSheet, Process, Cstr, TubularReactor,
)
from CADETProcess.simulator import Cadet

component_system = ComponentSystem(["Tracer"])
inlet = Inlet(component_system, "inlet")
inlet.c = [0]  # The pulse is set by events.
inlet.flow_rate = 1e-7  # 6 mL/min
outlet = Outlet(component_system, "outlet")
flow_sheet = FlowSheet(component_system, "flow_sheet")
flow_sheet.add_unit(inlet)
flow_sheet.add_unit(outlet)

Create a `Cstr` named `"mixer"` and a `TubularReactor` named `"tubing"` with the values below. Add both to `flow_sheet` with `add_unit()` and connect inlet → mixer → tubing → outlet with `add_connection()`.

| Unit | Attribute | Value |
|---|---|---|
| `mixer` | `init_liquid_volume` | `1e-6` m³ (1 mL) |
| `mixer` | `const_solid_volume` | `0` |
| `tubing` | `length` | `1` m |
| `tubing` | `diameter` | from $V = 2 \times 10^{-6}$ m³: $d=\sqrt{4V/(\pi L)}$ |
| `tubing` | `axial_dispersion` | `1e-8` m²/s |

In [ ]:
# Mixer, tubing and connections


In [ ]:
# Simulate the 5 s pulse and compare the detector signal with the measured pulse.
process = Process(flow_sheet, "python_system")
process.cycle_time = 300
process.add_event("pulse_on", "flow_sheet.inlet.c", 1, 0)
process.add_event("pulse_off", "flow_sheet.inlet.c", 0, 5)
simulator = Cadet()
simulator.time_resolution = 0.5  # store results every 0.5 s, like the measurement
simulation_results = simulator.simulate(process)

detector = simulation_results.solution.outlet.outlet
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(detector.time, detector.solution[:, 0], label="Python model")  # column 0: Tracer
ax.plot(pulse_data[:, 0], pulse_data[:, 1], "k--", label="measured system pulse")
ax.set(xlabel="Time / s", ylabel="Tracer / mol/m³", xlim=(0, 150))
_ = ax.legend()
plt.show()

```{dropdown} Answer
The two curves lie on top of each other: the synthetic measurement was generated with the same 1 mL mixer, 2 mL tubing and dispersion. With real data, differences between the curves show where the instrument model needs other volumes or dispersion.
```

**Next lesson:** [**02.5 Binding**](../02.5%20Binding/02.5a_binding_lesson.ipynb)